# 01 — Train MODEL A: GIÁ CƠ BẢN

**Vai trò:** đây là **lõi của kiến trúc Hybrid** đã chốt ở tuần 2:
```
giá cuối dự đoán = MODEL A (giá cơ bản) × MODEL B (hệ số nhân)
```

**Target:** `base_price = target_shown_price ÷ target_shown_multiplier` — phần giá **chưa nhân surge**,
do quãng đường + thời lượng quyết định (~92% theo permutation importance).

**Vì sao tách khỏi hệ số nhân:** giá cơ bản và hệ số nhân do **2 nhóm yếu tố hoàn toàn khác nhau**
quyết định (giá cơ bản ← thuộc tính chuyến đi; hệ số nhân ← cung–cầu). Tách 2 model để mỗi model học
đúng phần việc của nó — đã chứng minh Hybrid thắng dự đoán trực tiếp (MAE 18.048 vs 18.834).

**Notebook này CHỈ train Model A.** Việc ghép Hybrid + so sánh nằm ở `evaluation/04_eval_hybrid.ipynb`.

Các bước: (1) nạp dữ liệu · (2) cấu hình 3 thuật toán · (3) huấn luyện theo tháng · (4) kết quả sơ bộ · (5) lưu.

## 1. Nạp dữ liệu & tạo target

`base_price` và `latest_observed_base` là **cột dẫn xuất** — tính tại đây, không có sẵn trong parquet.

In [ ]:
import warnings, time, sys
from pathlib import Path
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import joblib
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from _common_train import CAT, B_NUM, dat_categories, prep, ALGOS, metrics
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

TARGET = "base_price"
NUM = B_NUM
LOG = True   # gia lech phai -> log-target

PREP = Path("../../data/hcm_train_ready.parquet")
assert PREP.exists(), "Chua co hcm_train_ready.parquet -> chay 00_chuan_bi_du_lieu.ipynb truoc!"
COLS = list(dict.fromkeys(CAT + B_NUM + ["target_shown_price", "target_shown_multiplier",
        "latest_observed_price", "latest_observed_multiplier", "evaluation_month", "split"]))
COLS = [c for c in COLS if c != "latest_observed_base"]   # cot dan xuat, khong co san
df = pd.read_parquet(PREP, columns=COLS)

# BAT BUOC: co dinh danh muc categorical tren df DAY DU truoc khi chia train/test
# (neu de prep() tu cast tung tap con, train/test co the co ma so khac nhau -> du doan sai)
df = dat_categories(df)

df["base_price"] = df.target_shown_price / df.target_shown_multiplier.clip(lower=0.1)
df["latest_observed_base"] = df.latest_observed_price / df.latest_observed_multiplier.clip(lower=0.1)

print(f"Nap {len(df):,} dong x {len(COLS)} cot (chi nap cot can dung, do RAM)")
print(f"Target={TARGET} | {len(CAT)+len(NUM)} feature | log-target={LOG}")
print(f"base_price median = {df.base_price.median():,.0f} VND")

## 2. Cấu hình 3 thuật toán

| Thuật toán | Siêu tham số chính | Ghi chú |
|---|---|---|
| **HistGB** (sklearn) | max_iter=500, learning_rate=0.05, l2=1.0, early_stopping | Mặc định — không cần thư viện ngoài sklearn |
| **LightGBM** | n_estimators=800, learning_rate=0.03, num_leaves=63 | Nhanh nhất trên dữ liệu lớn |
| **XGBoost** | n_estimators=800, learning_rate=0.03, max_depth=7 | enable_categorical=True |

Cả 3 dùng **chung một bộ feature** và cùng cách xử lý categorical (dtype `category`) → so sánh công bằng.

> Đã kiểm chứng ở tuần 2: 3 thuật toán cho kết quả chênh nhau **< 0,1%** (nằm trong nhiễu ngẫu nhiên),
> và fine-tune Optuna (40 trial/tháng) không cải thiện được (+2 VND). Siêu tham số dưới đây đã gần tối ưu.

In [ ]:
for algo, tao in ALGOS.items():
    print(f"[{algo}]\n  {tao()}\n")

## 3. Huấn luyện theo từng tháng (3 cell riêng — mỗi thuật toán 1 cell)

> ⚠️ **Train riêng từng tháng** (`evaluation_month`) — lịch sử giá đối thủ reset theo tháng, gộp sẽ rò rỉ.

LightGBM/XGBoost tách 10% train làm validation nội bộ để in loss + early-stop (giống cơ chế
`validation_fraction=0.1` có sẵn của HistGB) — **không đụng** vào `split=test`.

In [ ]:
models = {algo: {} for algo in ALGOS}
tests = {algo: {} for algo in ALGOS}
thangs = sorted(df.evaluation_month.unique())
print("Train theo thang:", thangs)

### 3a. HistGB

In [ ]:
print("=== HistGB ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    m = ALGOS["HistGB"]()
    ytr = np.log(tr[TARGET]) if LOG else tr[TARGET]
    m.fit(prep(tr, NUM), ytr)
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["HistGB"][th] = m; tests["HistGB"][th] = te
    tr_loss = -m.train_score_[-1]; val_loss = -m.validation_score_[-1]
    print(f"  [{th}] train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}  so_cay={m.n_iter_}  | {time.time()-t0:.1f}s")

### 3b. LightGBM

In [ ]:
print("=== LightGBM ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["LightGBM"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], eval_names=["train", "valid"],
          eval_metric="rmse", callbacks=[lgb.early_stopping(20, verbose=False)])
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["LightGBM"][th] = m; tests["LightGBM"][th] = te
    tr_loss = m.evals_result_["train"]["rmse"][-1]; val_loss = m.evals_result_["valid"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration_}  | {time.time()-t0:.1f}s")

### 3c. XGBoost

In [ ]:
print("=== XGBoost ===")
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()
    ytr_full = np.log(tr[TARGET]) if LOG else tr[TARGET]
    Xtr, Xval, ytr, yval = train_test_split(prep(tr, NUM), ytr_full, test_size=0.1, random_state=42)
    m = ALGOS["XGBoost"]()
    m.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xval, yval)], verbose=False)
    pred = m.predict(prep(te, NUM))
    te["base_pred"] = np.exp(pred) if LOG else pred
    models["XGBoost"][th] = m; tests["XGBoost"][th] = te
    ev = m.evals_result()
    tr_loss = ev["validation_0"]["rmse"][-1]; val_loss = ev["validation_1"]["rmse"][-1]
    print(f"  [{th}] train_rmse={tr_loss:.4f}  val_rmse={val_loss:.4f}  so_cay={m.best_iteration}  | {time.time()-t0:.1f}s")

## 4. Kết quả sơ bộ — so sánh 3 thuật toán

In cả bảng tổng gộp lẫn từng test-set nhỏ theo tháng (để thấy độ ổn định qua các tháng).

In [ ]:
allte = {algo: pd.concat(tests[algo].values()) for algo in ALGOS}

rows = []
for algo in ALGOS:
    d = allte[algo]
    rows.append({"Thuat toan": algo, "Test-set": "TAT CA", "n": len(d), **metrics(d[TARGET], d.base_pred)})
    for th in thangs:
        dt = d[d.evaluation_month==th]
        rows.append({"Thuat toan": algo, "Test-set": th, "n": len(dt), **metrics(dt[TARGET], dt.base_pred)})
bang = pd.DataFrame(rows).round(2)
print("MODEL A — GIA CO BAN (tong gop):"); display(bang[bang["Test-set"]=="TAT CA"].sort_values("MAE"))
print("\nTung test-set nho theo thang:"); display(bang[bang["Test-set"]!="TAT CA"])
print("\n(Tham chieu tuan 2: HistGB MAE ~15.032 VND | R2 ~0,656 | MAPE ~14,6%)")

## 5. Lưu model + dự đoán

- Model → `../<TenThuatToan>/gia_co_ban.joblib`
- Dự đoán test → `../evaluation/pred_gia_co_ban.parquet` (để notebook evaluation nạp lại, không train lại)

In [ ]:
for algo in ALGOS:
    Path(f"../{algo}").mkdir(exist_ok=True)
    joblib.dump(models[algo], f"../{algo}/gia_co_ban.joblib")
    print(f"Da luu ../{algo}/gia_co_ban.joblib")

giu = ["evaluation_month", "base_price", "base_pred", "target_shown_price", "target_shown_multiplier"]
pred_out = pd.concat([allte[algo][giu].assign(algo=algo) for algo in ALGOS], ignore_index=True)
Path("../evaluation").mkdir(exist_ok=True)
pred_out.to_parquet("../evaluation/pred_gia_co_ban.parquet", index=False)
print(f"\nDa luu ../evaluation/pred_gia_co_ban.parquet ({len(pred_out):,} dong)")
print("=> Buoc tiep: train/02_train_he_so_nhan.ipynb")